In [10]:
from helper_chat import chat, add_user_message, add_assistant_message

DATASET_FILENAME = 'dataset_eval.json'

In [11]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. 
The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. 
Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex",
    "solution_criteria" : "key criteria for evaluating a solution."
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages,stop_sequences=["```"])
    return json.loads(text)

In [12]:
dataset = generate_dataset()
dataset

with open(DATASET_FILENAME, "w") as f:
    json.dump(dataset, f, indent=2)


In [ ]:
def run_prompt(test_case):
    """ Merges the test case with the prompt and runs the prompt """
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}

    * Respond only with Python, JSON, or a plain Regex.
    * Do not add any comments or commentary or explanation.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    answer = chat(messages, temperature=0.0, stop_sequences=["```"])
    return answer



In [14]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.
    
Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}} 
"""
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    #print(eval_text)
    return json.loads(eval_text)

In [15]:
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [16]:
def run_test_case(test_case):
    """ calls run_prompt and grades the result """
    output = run_prompt(test_case)
    #Grading
    model_grade = grade_by_model(test_case,output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax (output,test_case)
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [17]:
from statistics import mean

def run_eval(dataset):
    """ Loads the dataset and call run_test_casewith each case """
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average Score {average_score}")
    return results  
    

In [18]:
with open(DATASET_FILENAME, "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average Score 6.833333333333333
[
  {
    "output": "\nimport re\nimport json\n\ndef parse_cloudwatch_log(log_entry):\n    pattern = r'\\[([^\\]]+)\\]\\s+\\[([^\\]]+)\\]\\s+(.+)'\n    match = re.match(pattern, log_entry)\n    \n    if match:\n        return {\n            \"timestamp\": match.group(1),\n            \"level\": match.group(2),\n            \"message\": match.group(3)\n        }\n    return None\n\nlog = \"[2024-01-15T10:30:45Z] [ERROR] Database connection failed\"\nresult = parse_cloudwatch_log(log)\nprint(json.dumps(result, indent=2))\n",
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message. The log format is: '[TIMESTAMP] [LEVEL] MESSAGE'",
      "format": "regex",
      "solution_criteria": "The regex should correctly capture three groups: timestamp (ISO 8601 format), log level (ERROR, WARN, INFO, DEBUG), and the message text. Should handle multi-word messages."
    },
    "score": 3.5,
    "reasoning": 